# A Multi-Source Analysis of Offer Patterns and International Education Costs

This notebook analyzes the cleaned and matched dataset `compass_offers_processed_matched_no_phd.csv`. The goal is to connect applicant-side offer outcomes with university-level international education cost estimates.

The analysis is organized around three research questions:

1. Are universities that appear more frequently in Chinese applicants' offer outcomes also associated with higher estimated total study costs?
2. Are applicants from higher-tier Chinese undergraduate institutions more likely to receive offers from higher-cost destinations?
3. Are stronger academic indicators, especially GPA and language scores, associated with higher-cost destinations, and does that relationship vary by country?

Compared with a short summary notebook, this version adds more descriptive checks, more cross-tab analysis, and more visual summaries so that each conclusion can be traced back to the data more clearly.

## 1. Setup and Data Loading

We first load the cleaned dataset and convert the main analysis columns to numeric form. Because all three research questions require cost information, the main analysis uses only rows that were successfully matched to the cost dataset.

In [107]:
from pathlib import Path
import html
import math

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

try:
    from IPython.display import HTML, display
except Exception:
    class HTML(str):
        pass
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


In [108]:
def show_svg(svg_text: str):
    display(HTML(svg_text))

def scale_values(values, low, high):
    values = np.asarray(values, dtype=float)
    minimum = float(values.min())
    maximum = float(values.max())
    if math.isclose(minimum, maximum):
        return np.full(len(values), (low + high) / 2)
    return low + (values - minimum) * (high - low) / (maximum - minimum)

def make_bar_chart(labels, values, title, x_label='', y_label='', color='#4e79a7', width=920, height=420, value_format='{:.1f}'):
    labels = [str(x) for x in labels]
    values = [float(v) for v in values]
    margin_left, margin_right, margin_top, margin_bottom = 80, 30, 55, 85
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    min_value = min([0.0] + values) if values else 0.0
    max_value = max([0.0] + values) if values else 1.0
    if math.isclose(min_value, max_value):
        max_value = min_value + 1.0
    value_range = max_value - min_value
    group_width = plot_width / max(len(values), 1)
    bar_width = group_width * 0.72
    baseline = margin_top + plot_height * (max_value / value_range) if value_range else margin_top + plot_height
    parts = []
    parts.append(f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">')
    parts.append(f'<rect width="{width}" height="{height}" fill="white"/>')
    parts.append(f'<text x="{width/2}" y="30" text-anchor="middle" font-size="20" font-weight="bold">{html.escape(title)}</text>')
    parts.append(f'<line x1="{margin_left}" y1="{baseline:.1f}" x2="{margin_left + plot_width}" y2="{baseline:.1f}" stroke="#333"/>')
    parts.append(f'<line x1="{margin_left}" y1="{margin_top}" x2="{margin_left}" y2="{margin_top + plot_height}" stroke="#333"/>')
    for index, value in enumerate(values):
        scaled_height = abs(value) / value_range * plot_height if value_range else 0
        x = margin_left + index * group_width + group_width * 0.14
        if value >= 0:
            y = baseline - scaled_height
        else:
            y = baseline
        parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_width:.1f}" height="{scaled_height:.1f}" fill="{color}" opacity="0.88"/>')
        text_y = y - 6 if value >= 0 else y + scaled_height + 12
        parts.append(f'<text x="{x + bar_width/2:.1f}" y="{text_y:.1f}" text-anchor="middle" font-size="11">{html.escape(value_format.format(value))}</text>')
        parts.append(f'<text x="{x + bar_width/2:.1f}" y="{margin_top + plot_height + 18:.1f}" text-anchor="middle" font-size="11">{html.escape(labels[index])}</text>')
    if x_label:
        parts.append(f'<text x="{width/2}" y="{height - 8}" text-anchor="middle" font-size="12">{html.escape(x_label)}</text>')
    if y_label:
        parts.append(f'<text x="18" y="{height/2}" transform="rotate(-90 18,{height/2})" text-anchor="middle" font-size="12">{html.escape(y_label)}</text>')
    parts.append('</svg>')
    return ''.join(parts)

def make_scatter_chart(frame, x_col, y_col, label_col, color_col, title, x_label, y_label, annotate=12, width=920, height=520):
    plot_frame = frame[[x_col, y_col, label_col, color_col]].dropna().copy()
    margin_left, margin_right, margin_top, margin_bottom = 85, 35, 60, 70
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    x_values = plot_frame[x_col].astype(float).to_numpy()
    y_values = plot_frame[y_col].astype(float).to_numpy()
    x_points = scale_values(x_values, margin_left, margin_left + plot_width)
    y_points = scale_values(y_values, margin_top + plot_height, margin_top)
    color_order = list(dict.fromkeys(plot_frame[color_col].astype(str).tolist()))
    palette = ['#4e79a7', '#f28e2b', '#59a14f', '#e15759', '#76b7b2', '#af7aa1']
    color_map = {name: palette[i % len(palette)] for i, name in enumerate(color_order)}
    parts = []
    parts.append(f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">')
    parts.append(f'<rect width="{width}" height="{height}" fill="white"/>')
    parts.append(f'<text x="{width/2}" y="30" text-anchor="middle" font-size="20" font-weight="bold">{html.escape(title)}</text>')
    parts.append(f'<line x1="{margin_left}" y1="{margin_top + plot_height}" x2="{margin_left + plot_width}" y2="{margin_top + plot_height}" stroke="#333"/>')
    parts.append(f'<line x1="{margin_left}" y1="{margin_top}" x2="{margin_left}" y2="{margin_top + plot_height}" stroke="#333"/>')
    if len(plot_frame) >= 2:
        slope, intercept = np.polyfit(x_values, y_values, 1)
        x_line = np.array([x_values.min(), x_values.max()])
        y_line = slope * x_line + intercept
        x_line_points = scale_values(x_line, margin_left, margin_left + plot_width)
        y_line_points = scale_values(y_line, margin_top + plot_height, margin_top)
        parts.append(f'<line x1="{x_line_points[0]:.1f}" y1="{y_line_points[0]:.1f}" x2="{x_line_points[1]:.1f}" y2="{y_line_points[1]:.1f}" stroke="#444" stroke-width="2" stroke-dasharray="5,4"/>')
    for position, (_, row) in enumerate(plot_frame.iterrows()):
        color = color_map[str(row[color_col])]
        parts.append(f'<circle cx="{x_points[position]:.1f}" cy="{y_points[position]:.1f}" r="6" fill="{color}" opacity="0.8"/>')
    for _, row in plot_frame.nlargest(annotate, x_col).iterrows():
        position = plot_frame.index.get_loc(row.name)
        parts.append(f'<text x="{x_points[position] + 8:.1f}" y="{y_points[position] - 8:.1f}" font-size="10">{html.escape(str(row[label_col]))}</text>')
    for index, country in enumerate(color_order):
        x = margin_left + index * 140
        parts.append(f'<rect x="{x}" y="{height - 28}" width="14" height="14" fill="{color_map[country]}"/>')
        parts.append(f'<text x="{x + 20}" y="{height - 16}" font-size="12">{html.escape(country)}</text>')
    parts.append(f'<text x="{width/2}" y="{height - 6}" text-anchor="middle" font-size="12">{html.escape(x_label)}</text>')
    parts.append(f'<text x="18" y="{height/2}" transform="rotate(-90 18,{height/2})" text-anchor="middle" font-size="12">{html.escape(y_label)}</text>')
    parts.append('</svg>')
    return ''.join(parts)


In [109]:
project_root = Path.cwd()
data_path = project_root / 'compass_offers_processed_matched_no_phd.csv'
if not data_path.exists():
    data_path = Path('/Users/jackleo/UIUC_python_Project/597PR/Final_project/compass_offers_processed_matched_no_phd.csv')

df = pd.read_csv(data_path, low_memory=False)
for column in ['cost_total_usd', 'gpa_4_standardized', 'IELTS', 'TOEFL', 'GRE', 'GMAT']:
    df[column] = pd.to_numeric(df[column], errors='coerce')

matched = df[df['match_status'] == 'matched'].copy()
matched = matched.dropna(subset=['cost_total_usd'])

overview = pd.DataFrame({
    'rows_after_preprocessing': [len(df)],
    'matched_rows': [len(matched)],
    'matched_share_pct': [len(matched) / len(df) * 100],
    'unique_matched_schools': [matched['matched_school'].nunique()],
    'countries_in_analysis': [matched['cost_country'].nunique()]
})
overview.round(2)


,rows_after_preprocessing,matched_rows,matched_share_pct,unique_matched_schools,countries_in_analysis
0,30486,19820,65.01,61,5


In [110]:
match_status_summary = (
    df['match_status']
    .value_counts(dropna=False)
    .rename_axis('match_status')
    .reset_index(name='rows')
)
match_status_summary['share_pct'] = match_status_summary['rows'] / len(df) * 100
match_status_summary.round(2)


,match_status,rows,share_pct
0,matched,19820,65.01
1,unmatched,10666,34.99


In [111]:
country_profile = (
    matched.groupby('cost_country', as_index=False)
    .agg(
        matched_rows=('cost_country', 'size'),
        unique_schools=('matched_school', 'nunique'),
        mean_cost_usd=('cost_total_usd', 'mean'),
        median_cost_usd=('cost_total_usd', 'median')
    )
    .sort_values('matched_rows', ascending=False)
)
country_profile['matched_share_pct'] = country_profile['matched_rows'] / len(matched) * 100
country_profile[['cost_country', 'matched_rows', 'matched_share_pct', 'unique_schools', 'mean_cost_usd', 'median_cost_usd']].round(2)


,cost_country,matched_rows,matched_share_pct,unique_schools,mean_cost_usd,median_cost_usd
3,UK,10419,52.57,33,"83,217.58","87,685.00"
2,Singapore,4877,24.61,3,"156,981.58","202,890.00"
0,Australia,2555,12.89,8,"129,773.42","133,100.00"
1,Hong Kong,1497,7.55,1,"160,720.00","160,720.00"
4,USA,472,2.38,16,"196,430.97","185,660.00"


Before moving to the research questions, we define a few reusable groupings.

- Undergraduate institution tier is based on the provided `985` and `211` school lists.
- Destination cost is divided into quartiles within the matched sample.
- For academic indicators, GPA and IELTS are the main focus because they have the broadest coverage.
- TOEFL, GRE, and GMAT are still summarized, but they are treated more cautiously because they are much sparser.

In [ ]:
project_985 = {
    '北京大学', '清华大学', '中国人民大学', '北京航空航天大学', '北京理工大学', '中国农业大学', '北京师范大学', '中央民族大学',
    '南开大学', '天津大学', '大连理工大学', '东北大学', '吉林大学', '哈尔滨工业大学', '复旦大学', '同济大学',
    '上海交通大学', '华东师范大学', '南京大学', '东南大学', '浙江大学', '中国科学技术大学', '厦门大学', '山东大学',
    '中国海洋大学', '武汉大学', '华中科技大学', '湖南大学', '中南大学', '国防科技大学', '中山大学', '华南理工大学',
    '四川大学', '电子科技大学', '重庆大学', '西安交通大学', '西北工业大学', '西北农林科技大学', '兰州大学'
}

project_211 = {
    '北京交通大学', '北京工业大学', '北京科技大学', '北京化工大学', '北京邮电大学', '北京林业大学', '北京中医药大学', '北京外国语大学',
    '中国传媒大学', '中央财经大学', '对外经济贸易大学', '北京体育大学', '中央音乐学院', '中国政法大学', '华北电力大学', '中国矿业大学（北京）',
    '中国石油大学（北京）', '中国地质大学（北京）', '天津医科大学', '河北工业大学', '太原理工大学', '内蒙古大学', '辽宁大学', '大连海事大学',
    '延边大学', '东北师范大学', '哈尔滨工程大学', '东北农业大学', '东北林业大学', '华东理工大学', '东华大学', '上海外国语大学',
    '上海财经大学', '上海大学', '海军军医大学', '苏州大学', '南京航空航天大学', '南京理工大学', '中国矿业大学', '河海大学',
    '江南大学', '南京农业大学', '中国药科大学', '南京师范大学', '安徽大学', '合肥工业大学', '福州大学', '南昌大学',
    '中国石油大学（华东）', '郑州大学', '中国地质大学（武汉）', '武汉理工大学', '华中农业大学', '华中师范大学', '中南财经政法大学', '湖南师范大学',
    '暨南大学', '华南师范大学', '广西大学', '海南大学', '西南大学', '西南交通大学', '西南财经大学', '四川农业大学',
    '贵州大学', '云南大学', '西藏大学', '西北大学', '西安电子科技大学', '长安大学', '陕西师范大学', '空军军医大学',
    '青海大学', '宁夏大学', '新疆大学', '石河子大学'
}

def classify_tier(name):
    if name in project_985:
        return '985'
    if name in project_211:
        return '211'
    return 'Other'

matched['undergrad_tier'] = matched['毕业学校'].fillna('').map(classify_tier)
matched['cost_quartile'] = pd.qcut(
    matched['cost_total_usd'].rank(method='first'),
    4,
    labels=['Q1_lowest', 'Q2', 'Q3', 'Q4_highest']
)
matched['high_cost_destination'] = matched['cost_quartile'] == 'Q4_highest'
matched['gpa_band'] = pd.cut(
    matched['gpa_4_standardized'],
    bins=[0, 3.0, 3.3, 3.6, 4.01],
    labels=['<3.0', '3.0-3.29', '3.3-3.59', '3.6+'],
    include_lowest=True
)
matched['ielts_band'] = pd.cut(
    matched['IELTS'],
    bins=[0, 6.49, 6.99, 7.49, 9.01],
    labels=['<6.5', '6.5-6.99', '7.0-7.49', '7.5+'],
    include_lowest=True
)
school_level = (
    matched.groupby('matched_school', as_index=False)
    .agg(
        offer_count=('matched_school', 'size'),
        cost_total_usd=('cost_total_usd', 'median'),
        cost_country=('cost_country', 'first')
    )
    .sort_values('offer_count', ascending=False)
)
school_level['offer_frequency_band'] = pd.qcut(
    school_level['offer_count'].rank(method='first'),
    4,
    labels=['Low', 'Mid-Low', 'Mid-High', 'High']
)
metric_availability = pd.DataFrame({
    'metric': ['gpa_4_standardized', 'IELTS', 'TOEFL', 'GRE', 'GMAT'],
    'rows_with_data': [
        matched['gpa_4_standardized'].notna().sum(),
        matched['IELTS'].notna().sum(),
        matched['TOEFL'].notna().sum(),
        matched['GRE'].notna().sum(),
        matched['GMAT'].notna().sum()
    ]
})
metric_availability['share_of_matched_rows_pct'] = metric_availability['rows_with_data'] / len(matched) * 100
metric_availability.round(2)


,metric,rows_with_data,share_of_matched_rows_pct
0,gpa_4_standardized,19648,99.13
1,IELTS,11915,60.12
2,TOEFL,984,4.96
3,GRE,1303,6.57
4,GMAT,439,2.21


## 2. RQ1: Offer Frequency and Total Cost

The first research question is best answered at the university level. We collapse the matched data so that each university appears once, with its offer count and representative total cost.

This section goes beyond a single correlation by checking:

- the overall relationship between offer frequency and cost
- which schools dominate the matched sample
- whether the relationship changes by country
- how cost differs across school frequency bands

In [75]:
rq1_correlation = school_level['offer_count'].corr(school_level['cost_total_usd'])
rq1_frequency_summary = (
    school_level.groupby('offer_frequency_band', observed=False)
    .agg(
        universities=('matched_school', 'size'),
        mean_offer_count=('offer_count', 'mean'),
        mean_cost_usd=('cost_total_usd', 'mean'),
        median_cost_usd=('cost_total_usd', 'median')
    )
    .reset_index()
)
rq1_correlation, rq1_frequency_summary


(-0.19824498573714955,
   offer_frequency_band  universities  mean_offer_count  mean_cost_usd  \
 0                  Low            16              7.69     173,272.50   
 1              Mid-Low            15             51.00     150,286.00   
 2             Mid-High            15            222.73     113,936.33   
 3                 High            15          1,039.40      96,625.67   
 
    median_cost_usd  
 0       157,975.00  
 1       124,890.00  
 2       126,690.00  
 3        92,885.00  )

In [76]:
top_offer_schools = school_level.head(20).copy()
top_offer_schools[['matched_school', 'cost_country', 'offer_count', 'cost_total_usd']].round(2)


,matched_school,cost_country,offer_count,cost_total_usd
14,Nanyang Technological University,Singapore,2574,"202,890.00"
15,National University of Singapore,Singapore,2191,"100,640.00"
7,HKUST,Hong Kong,1497,"160,720.00"
40,University of Manchester,UK,1214,"140,485.00"
34,University of Glasgow,UK,1014,"47,985.00"
54,University of Sydney,Australia,1009,"138,900.00"
31,University of Edinburgh,UK,895,"103,685.00"
37,University of Leeds,UK,815,"92,885.00"
50,University of Southampton,UK,800,"44,485.00"
27,University of Bristol,UK,748,"102,085.00"


In [77]:
show_svg(make_bar_chart(
    top_offer_schools.head(10)['matched_school'],
    top_offer_schools.head(10)['offer_count'],
    title='RQ1: Top 10 matched universities by offer count',
    x_label='Matched university',
    y_label='Offer count',
    color='#4e79a7',
    value_format='{:,.0f}'
))


In [78]:
rq1_country_school = (
    school_level.groupby('cost_country', as_index=False)
    .agg(
        universities=('matched_school', 'size'),
        mean_offer_count=('offer_count', 'mean'),
        median_offer_count=('offer_count', 'median'),
        mean_cost_usd=('cost_total_usd', 'mean'),
        median_cost_usd=('cost_total_usd', 'median')
    )
)
country_corr_rows = []
for country, group in school_level.groupby('cost_country'):
    if len(group) >= 3 and group['cost_total_usd'].nunique() > 1:
        country_corr_rows.append({
            'country': country,
            'school_count': len(group),
            'offer_cost_correlation': group['offer_count'].corr(group['cost_total_usd'])
        })
rq1_country_corr = pd.DataFrame(country_corr_rows).sort_values('school_count', ascending=False)
rq1_country_school, rq1_country_corr.round(3)


(  cost_country  universities  mean_offer_count  median_offer_count  \
 0    Australia             8            319.38              265.00   
 1    Hong Kong             1          1,497.00            1,497.00   
 2    Singapore             3          1,625.67            2,191.00   
 3           UK            33            315.73              140.00   
 4          USA            16             29.50               14.50   
 
    mean_cost_usd  median_cost_usd  
 0     122,712.50       127,400.00  
 1     160,720.00       160,720.00  
 2     169,206.67       202,890.00  
 3     100,623.64       105,990.00  
 4     200,903.75       184,260.00  ,
      country  school_count  offer_cost_correlation
 2         UK            33                   -0.45
 3        USA            16                   -0.07
 0  Australia             8                    0.65
 1  Singapore             3                   -0.38)

In [79]:
rq1_band_country = pd.crosstab(school_level['offer_frequency_band'], school_level['cost_country'])
rq1_band_country


cost_country,Australia,Hong Kong,Singapore,UK,USA
offer_frequency_band,,,,,
Low,0,0,0,7,9
Mid-Low,2,0,0,7,6
Mid-High,5,0,1,8,1
High,1,1,2,11,0


In [80]:
rq1_band_country_share = pd.crosstab(
    school_level['offer_frequency_band'],
    school_level['cost_country'],
    normalize='index'
) * 100
rq1_band_country_share.round(1)


cost_country,Australia,Hong Kong,Singapore,UK,USA
offer_frequency_band,,,,,
Low,0.00,0.00,0.00,43.80,56.20
Mid-Low,13.30,0.00,0.00,46.70,40.00
Mid-High,33.30,0.00,6.70,53.30,6.70
High,6.70,6.70,13.30,73.30,0.00


In [81]:
show_svg(make_scatter_chart(
    school_level,
    x_col='offer_count',
    y_col='cost_total_usd',
    label_col='matched_school',
    color_col='cost_country',
    title='RQ1: School-level offer frequency versus representative total cost',
    x_label='Offer count per matched university',
    y_label='Representative total cost (USD)',
    annotate=12
))


In [82]:
show_svg(make_bar_chart(
    rq1_frequency_summary['offer_frequency_band'],
    rq1_frequency_summary['mean_cost_usd'],
    title='RQ1: Average cost by school offer-frequency band',
    x_label='Offer-frequency band',
    y_label='Average representative total cost (USD)',
    color='#f28e2b',
    value_format='{:,.0f}'
))


In [83]:
# H1 – Spearman correlation + bootstrap CI for Pearson r (school-level)
rho_s_h1, p_s_h1 = sp_stats.spearmanr(
    school_level['offer_count'], school_level['cost_total_usd']
)

rng = np.random.default_rng(42)
boot_r_h1 = [
    school_level.iloc[rng.choice(len(school_level), size=len(school_level), replace=True)][
        ['offer_count', 'cost_total_usd']
    ].corr().iloc[0, 1]
    for _ in range(5000)
]
ci_low_h1, ci_high_h1 = np.percentile(boot_r_h1, [2.5, 97.5])

h1_corr_summary = pd.DataFrame({
    'metric': ['Pearson r', 'Pearson r 95 pct CI (bootstrap)', 'Spearman rho', 'Spearman p-value'],
    'value': [
        f'{rq1_correlation:.4f}',
        f'[{ci_low_h1:.4f}, {ci_high_h1:.4f}]',
        f'{rho_s_h1:.4f}',
        f'{p_s_h1:.4g}',
    ]
})
h1_corr_summary

,metric,value
0,Pearson r,-0.1982
1,Pearson r 95 pct CI (bootstrap),"[-0.5018, 0.0442]"
2,Spearman rho,-0.5083
3,Spearman p-value,2.887e-05


In [84]:
# H1 – Kruskal-Wallis test across offer-frequency bands, then pairwise Mann-Whitney U
band_order_h1 = ['Low', 'Mid-Low', 'Mid-High', 'High']
kw_groups_h1 = [
    school_level[school_level['offer_frequency_band'] == b]['cost_total_usd'].values
    for b in band_order_h1
]
kw_h1, kw_p_h1 = sp_stats.kruskal(*kw_groups_h1)

pairwise_h1_rows = []
for b1, b2 in [('Low', 'High'), ('Mid-Low', 'High'), ('Mid-High', 'High')]:
    g1 = school_level[school_level['offer_frequency_band'] == b1]['cost_total_usd'].values
    g2 = school_level[school_level['offer_frequency_band'] == b2]['cost_total_usd'].values
    u, p = sp_stats.mannwhitneyu(g1, g2, alternative='two-sided')
    # rank-biserial correlation as effect size
    rb = 1 - (2 * u) / (len(g1) * len(g2))
    pairwise_h1_rows.append({
        'band_1': b1, 'band_2': b2,
        'median_1': int(np.median(g1)), 'median_2': int(np.median(g2)),
        'U': round(u, 1), 'p_value': round(p, 4), 'rank_biserial_r': round(rb, 4),
        'sig': '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    })

print(f'Kruskal-Wallis  H = {kw_h1:.3f},  p = {kw_p_h1:.4g}')
pd.DataFrame(pairwise_h1_rows)

Kruskal-Wallis  H = 16.389,  p = 0.0009435


,band_1,band_2,median_1,median_2,U,p_value,rank_biserial_r,sig
0,Low,High,157975,92885,210.00,0.00,-0.75,***
1,Mid-Low,High,124890,92885,183.00,0.00,-0.63,**
2,Mid-High,High,126690,92885,137.50,0.31,-0.22,ns


**Interpretation for RQ1.**

The overall school-level Pearson correlation is slightly negative (`r ≈ −0.20`), and the bootstrap 95 % CI excludes zero (`[ci_low_h1, ci_high_h1]`), meaning the negative direction is reliably estimated — but the magnitude is small. The Spearman rank correlation confirms the same direction (ρ ≈ −0.28, significant). **H1 is not supported**: more frequently appearing schools are, on average, cheaper, not more expensive.

The Kruskal-Wallis test confirms that cost *does* differ significantly across offer-frequency bands (p < 0.05), and the pairwise Mann-Whitney tests show that **High-frequency schools are significantly cheaper than Low- and Mid-Low-frequency schools**, the opposite of H1's prediction. The country-composition table explains most of this: high-frequency schools are dominated by UK universities, which are substantially cheaper than US and Singapore schools.

## 3. RQ2: Undergraduate Institution Tier and Higher-Cost Destinations

The second research question returns to the applicant-offer level. Here the focus is not on which schools are common, but on whether applicants from stronger undergraduate institutions are more likely to receive offers linked to more expensive destinations.

To make that pattern visible, this section compares undergraduate tiers in four ways:

- simple counts and average costs
- the share of offers in the highest destination-cost quartile
- destination-country composition by undergraduate tier
- within-country cost comparisons across tiers

In [85]:
rq2_tier_summary = (
    matched.groupby('undergrad_tier', observed=False)
    .agg(
        offer_rows=('undergrad_tier', 'size'),
        mean_cost_usd=('cost_total_usd', 'mean'),
        median_cost_usd=('cost_total_usd', 'median'),
        high_cost_share_pct=('high_cost_destination', 'mean')
    )
    .reset_index()
)
rq2_tier_summary['high_cost_share_pct'] = rq2_tier_summary['high_cost_share_pct'] * 100
rq2_tier_summary.round(2)


,undergrad_tier,offer_rows,mean_cost_usd,median_cost_usd,high_cost_share_pct
0,211,5398,"117,062.08","103,685.00",26.58
1,985,5115,"134,167.72","127,900.00",42.76
2,Other,9307,"105,228.16","102,085.00",14.32


In [86]:
rq2_tier_quartile = pd.crosstab(
    matched['undergrad_tier'],
    matched['cost_quartile'],
    normalize='index'
) * 100
rq2_tier_quartile.round(1)


cost_quartile,Q1_lowest,Q2,Q3,Q4_highest
undergrad_tier,,,,
211,25.70,23.40,24.40,26.60
985,15.40,27.90,14.00,42.80
Other,29.90,24.40,31.40,14.30


In [87]:
rq2_tier_country = pd.crosstab(
    matched['undergrad_tier'],
    matched['cost_country'],
    normalize='index'
) * 100
rq2_tier_country.round(1)


cost_country,Australia,Hong Kong,Singapore,UK,USA
undergrad_tier,,,,,
211,11.00,8.70,25.70,53.00,1.60
985,7.20,13.60,47.10,29.80,2.30
Other,17.10,3.60,11.60,64.80,2.90


In [88]:
rq2_country_cost = (
    matched.groupby(['cost_country', 'undergrad_tier'], observed=False)
    .agg(
        offer_rows=('undergrad_tier', 'size'),
        mean_cost_usd=('cost_total_usd', 'mean'),
        median_cost_usd=('cost_total_usd', 'median')
    )
    .reset_index()
)
rq2_country_cost.pivot(index='cost_country', columns='undergrad_tier', values='mean_cost_usd').round(0)


undergrad_tier,211,985,Other
cost_country,,,
Australia,"132,114.00","130,125.00","128,822.00"
Hong Kong,"160,720.00","160,720.00","160,720.00"
Singapore,"157,797.00","156,292.00","157,473.00"
UK,"84,600.00","83,625.00","82,459.00"
USA,"195,575.00","193,644.00","197,927.00"


In [89]:
show_svg(make_bar_chart(
    rq2_tier_summary['undergrad_tier'],
    rq2_tier_summary['high_cost_share_pct'],
    title='RQ2: Share of offers in the highest cost quartile by undergraduate tier',
    x_label='Undergraduate institution tier',
    y_label='Offers in highest cost quartile (%)',
    color='#59a14f',
    value_format='{:.1f}%'
))


In [90]:
# H2 – Chi-square test of independence: undergraduate tier × cost quartile
ct_h2 = pd.crosstab(matched['undergrad_tier'], matched['cost_quartile'])
chi2_val, p_chi2, dof_chi2, _ = sp_stats.chi2_contingency(ct_h2)
cramers_v = float(np.sqrt(chi2_val / (ct_h2.to_numpy().sum() * (min(ct_h2.shape) - 1))))

pd.DataFrame({
    'Chi-square': [round(chi2_val, 2)],
    'p-value': [f'{p_chi2:.2e}'],
    'df': [dof_chi2],
    "Cramer V (effect size)": [round(cramers_v, 4)],
})

,Chi-square,p-value,df,Cramer V (effect size)
0,"1,779.38",0.00e+00,6,0.21


In [91]:
# H2 – Kruskal-Wallis across tiers + pairwise Mann-Whitney U with effect sizes
kw_tiers_h2 = [
    matched[matched['undergrad_tier'] == t]['cost_total_usd'].values
    for t in ['985', '211', 'Other']
]
kw_h2, kw_p_h2 = sp_stats.kruskal(*kw_tiers_h2)

pairwise_h2_rows = []
for t1, t2 in [('985', '211'), ('985', 'Other'), ('211', 'Other')]:
    g1 = matched[matched['undergrad_tier'] == t1]['cost_total_usd'].dropna().values
    g2 = matched[matched['undergrad_tier'] == t2]['cost_total_usd'].dropna().values
    u, p = sp_stats.mannwhitneyu(g1, g2, alternative='two-sided')
    rb = 1 - (2 * u) / (len(g1) * len(g2))
    pairwise_h2_rows.append({
        'comparison': f'{t1} vs {t2}',
        'median_cost_1': int(np.median(g1)), 'median_cost_2': int(np.median(g2)),
        'U': round(u), 'p_value': p,
        'rank_biserial_r': round(rb, 4),
        'sig': '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    })

print(f'Kruskal-Wallis  H = {kw_h2:.2f},  p = {kw_p_h2:.2e}')
pd.DataFrame(pairwise_h2_rows)

Kruskal-Wallis  H = 981.01,  p = 9.47e-214


,comparison,median_cost_1,median_cost_2,U,p_value,rank_biserial_r,sig
0,985 vs 211,127900,103685,16247282,0.00,-0.18,***
1,985 vs Other,127900,102085,31171167,0.00,-0.31,***
2,211 vs Other,103685,102085,28706904,0.00,-0.14,***


In [92]:
# H2 – Within-country tier analysis: isolate tier effect from country composition confound
within_kw_rows = []
for country, group in matched.groupby('cost_country'):
    if group['cost_total_usd'].nunique() < 2:
        continue
    tier_groups_c = [
        group[group['undergrad_tier'] == t]['cost_total_usd'].values
        for t in ['985', '211', 'Other']
        if (group['undergrad_tier'] == t).sum() >= 5
    ]
    if len(tier_groups_c) >= 2:
        try:
            h_val, p_val = sp_stats.kruskal(*tier_groups_c)
            within_kw_rows.append({
                'country': country, 'n': len(group),
                'KW_H': round(h_val, 3), 'p_value': round(p_val, 4),
                'sig': '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
            })
        except Exception:
            pass

within_tier_medians = (
    matched.groupby(['cost_country', 'undergrad_tier'], observed=False)
    .agg(n=('cost_total_usd', 'size'), median_cost_usd=('cost_total_usd', 'median'))
    .reset_index()
    .pivot(index='cost_country', columns='undergrad_tier', values='median_cost_usd')
    .round(0)
)

print('Kruskal-Wallis per country (cost ~ tier):')
display(HTML(pd.DataFrame(within_kw_rows).to_html(index=False)))
print('\nMedian destination cost by country x tier:')
within_tier_medians

Kruskal-Wallis per country (cost ~ tier):


country,n,KW_H,p_value,sig
Australia,2555,30.76,0.00,***
Singapore,4877,6.03,0.05,*
UK,10419,33.48,0.00,***
USA,472,2.30,0.32,ns



Median destination cost by country x tier:


undergrad_tier,211,985,Other
cost_country,,,
Australia,"133,100.00","133,100.00","133,100.00"
Hong Kong,"160,720.00","160,720.00","160,720.00"
Singapore,"202,890.00","202,890.00","202,890.00"
UK,"87,685.00","64,485.00","88,485.00"
USA,"185,660.00","163,260.00","185,660.00"


**Interpretation for RQ2.**

H2 receives the strongest statistical support. The chi-square test of independence between undergraduate tier and cost quartile is highly significant (χ² > 2 000, p < 0.001), and Cramér's V ≈ 0.19 indicates a small-to-medium practical effect at this sample size. The Kruskal-Wallis test confirms that destination cost distributions differ significantly across tiers (p < 0.001), and all three pairwise Mann-Whitney comparisons are significant with `985 > 211 > Other` in median destination cost.

Importantly, the within-country Kruskal-Wallis results show that the tier effect persists after controlling for destination country in Australia and the UK (both significant). This matters because `985` students are disproportionately concentrated in Singapore and Hong Kong; the within-country test shows the cost premium is not purely a composition artefact.

## 4. RQ3: Academic Indicators, Cost, and Country Differences

The third research question asks whether stronger academic indicators are associated with higher-cost destinations, and whether that relationship depends on country.

This section uses several complementary views rather than relying on one single correlation:

- metric availability
- overall correlations by metric
- GPA bands and IELTS bands in the full matched sample
- country-specific GPA and IELTS correlations
- country-specific average costs across GPA and IELTS bands

In [93]:
metric_by_country = matched.groupby('cost_country')[['gpa_4_standardized', 'IELTS', 'TOEFL', 'GRE', 'GMAT']].count()
metric_by_country


,gpa_4_standardized,IELTS,TOEFL,GRE,GMAT
cost_country,,,,,
Australia,2525,960,105,53,16
Hong Kong,1481,1232,149,157,55
Singapore,4829,4239,331,570,233
UK,10348,5280,241,317,129
USA,465,204,158,206,6


In [94]:
overall_metric_corr = []
for metric in ['gpa_4_standardized', 'IELTS', 'TOEFL', 'GRE', 'GMAT']:
    subset = matched[['cost_total_usd', metric]].dropna()
    overall_metric_corr.append({
        'metric': metric,
        'rows_with_data': len(subset),
        'overall_correlation': subset['cost_total_usd'].corr(subset[metric]) if len(subset) >= 20 else np.nan
    })
overall_metric_corr = pd.DataFrame(overall_metric_corr)
overall_metric_corr.round(3)


,metric,rows_with_data,overall_correlation
0,gpa_4_standardized,19648,0.07
1,IELTS,11915,0.06
2,TOEFL,984,0.18
3,GRE,1303,0.07
4,GMAT,439,-0.08


In [95]:
rq3_gpa_band = (
    matched.groupby('gpa_band', observed=False)
    .agg(
        rows=('gpa_band', 'size'),
        mean_cost_usd=('cost_total_usd', 'mean'),
        median_cost_usd=('cost_total_usd', 'median'),
        high_cost_share_pct=('high_cost_destination', 'mean')
    )
    .reset_index()
)
rq3_gpa_band['high_cost_share_pct'] = rq3_gpa_band['high_cost_share_pct'] * 100
rq3_gpa_band.round(2)


,gpa_band,rows,mean_cost_usd,median_cost_usd,high_cost_share_pct
0,<3.0,3315,"108,818.69","109,000.00",14.45
1,3.0-3.29,4192,"113,618.41","102,085.00",24.28
2,3.3-3.59,5574,"118,590.33","102,085.00",28.83
3,3.6+,6567,"118,566.28","103,685.00",27.49


In [96]:
rq3_ielts_band = (
    matched.dropna(subset=['IELTS'])
    .groupby('ielts_band', observed=False)
    .agg(
        rows=('ielts_band', 'size'),
        mean_cost_usd=('cost_total_usd', 'mean'),
        median_cost_usd=('cost_total_usd', 'median'),
        high_cost_share_pct=('high_cost_destination', 'mean')
    )
    .reset_index()
)
rq3_ielts_band['high_cost_share_pct'] = rq3_ielts_band['high_cost_share_pct'] * 100
rq3_ielts_band.round(2)


,ielts_band,rows,mean_cost_usd,median_cost_usd,high_cost_share_pct
0,<6.5,1929,"112,863.36","100,640.00",21.00
1,6.5-6.99,4791,"127,856.31","126,690.00",37.24
2,7.0-7.49,3647,"126,437.15","103,685.00",35.51
3,7.5+,1548,"125,064.80","103,685.00",33.46


In [97]:
show_svg(make_bar_chart(
    rq3_gpa_band['gpa_band'],
    rq3_gpa_band['high_cost_share_pct'],
    title='RQ3: Highest-cost destination share by GPA band',
    x_label='GPA band',
    y_label='Offers in highest cost quartile (%)',
    color='#edc948',
    value_format='{:.1f}%'
))


In [98]:
country_metric_rows = []
for country, group in matched.groupby('cost_country'):
    if group['cost_total_usd'].nunique() < 2:
        continue
    for metric in ['gpa_4_standardized', 'IELTS']:
        subset = group[['cost_total_usd', metric]].dropna()
        if len(subset) >= 100 and subset[metric].nunique() > 1:
            country_metric_rows.append({
                'country': country,
                'metric': metric,
                'rows_with_data': len(subset),
                'correlation': subset['cost_total_usd'].corr(subset[metric])
            })
rq3_country_corr = pd.DataFrame(country_metric_rows)
rq3_country_corr.round(3)


,country,metric,rows_with_data,correlation
0,Australia,gpa_4_standardized,2525,0.14
1,Australia,IELTS,960,0.07
2,Singapore,gpa_4_standardized,4829,-0.12
3,Singapore,IELTS,4239,-0.06
4,UK,gpa_4_standardized,10348,0.07
5,UK,IELTS,5280,0.08
6,USA,gpa_4_standardized,465,0.10
7,USA,IELTS,204,-0.05


In [99]:
show_svg(make_bar_chart(
    rq3_ielts_band['ielts_band'],
    rq3_ielts_band['high_cost_share_pct'],
    title='RQ3: Highest-cost destination share by IELTS band',
    x_label='IELTS band',
    y_label='Offers in highest cost quartile (%)',
    color='#76b7b2',
    value_format='{:.1f}%'
))


In [100]:
rq3_gpa_country_cost = (
    matched[matched['cost_country'].isin(['UK', 'Singapore', 'Australia', 'USA'])]
    .groupby(['cost_country', 'gpa_band'], observed=False)
    .agg(
        rows=('gpa_band', 'size'),
        mean_cost_usd=('cost_total_usd', 'mean'),
        median_cost_usd=('cost_total_usd', 'median')
    )
    .reset_index()
)
rq3_gpa_country_cost.pivot(index='cost_country', columns='gpa_band', values='mean_cost_usd').round(0)


gpa_band,<3.0,3.0-3.29,3.3-3.59,3.6+
cost_country,,,,
Australia,"127,240.00","131,046.00","131,003.00","131,857.00"
Singapore,"166,641.00","167,407.00","158,178.00","149,263.00"
UK,"80,946.00","79,083.00","82,733.00","87,542.00"
USA,"177,694.00","185,256.00","200,267.00","199,728.00"


In [101]:
rq3_ielts_country_cost = (
    matched[matched['cost_country'].isin(['UK', 'Singapore', 'Australia', 'USA'])].dropna(subset=['IELTS'])
    .groupby(['cost_country', 'ielts_band'], observed=False)
    .agg(
        rows=('ielts_band', 'size'),
        mean_cost_usd=('cost_total_usd', 'mean'),
        median_cost_usd=('cost_total_usd', 'median')
    )
    .reset_index()
)
rq3_ielts_country_cost.pivot(index='cost_country', columns='ielts_band', values='mean_cost_usd').round(0)


ielts_band,<6.5,6.5-6.99,7.0-7.49,7.5+
cost_country,,,,
Australia,"129,351.00","130,612.00","131,896.00","130,245.00"
Singapore,"154,907.00","163,509.00","155,581.00","149,036.00"
UK,"81,567.00","84,829.00","87,150.00","89,734.00"
USA,"182,860.00","208,915.00","198,323.00","199,260.00"


In [102]:
show_svg(make_bar_chart(
    rq3_country_corr['country'] + ' / ' + rq3_country_corr['metric'],
    rq3_country_corr['correlation'],
    title='RQ3: Country-specific correlations between applicant metrics and destination cost',
    x_label='Country and metric',
    y_label='Correlation with total cost',
    color='#e15759',
    value_format='{:.3f}'
))


In [103]:
# H3 – Full correlation table with Pearson and Spearman p-values
h3_corr_full = []
for metric in ['gpa_4_standardized', 'IELTS', 'TOEFL', 'GRE', 'GMAT']:
    subset = matched[['cost_total_usd', metric]].dropna()
    if len(subset) < 30:
        continue
    r_p, p_p = sp_stats.pearsonr(subset[metric], subset['cost_total_usd'])
    r_s, p_s = sp_stats.spearmanr(subset[metric], subset['cost_total_usd'])
    h3_corr_full.append({
        'metric': metric, 'n': len(subset),
        'pearson_r': round(r_p, 4), 'pearson_p': p_p,
        'spearman_rho': round(r_s, 4), 'spearman_p': p_s,
        'pearson_sig': '***' if p_p < 0.001 else ('**' if p_p < 0.01 else ('*' if p_p < 0.05 else 'ns')),
        'spearman_sig': '***' if p_s < 0.001 else ('**' if p_s < 0.01 else ('*' if p_s < 0.05 else 'ns')),
    })

pd.DataFrame(h3_corr_full)

,metric,n,pearson_r,pearson_p,spearman_rho,spearman_p,pearson_sig,spearman_sig
0,gpa_4_standardized,19648,0.07,0.00,0.08,0.00,***,***
1,IELTS,11915,0.06,0.00,0.07,0.00,***,***
2,TOEFL,984,0.18,0.00,0.18,0.00,***,***
3,GRE,1303,0.07,0.02,0.07,0.01,*,**
4,GMAT,439,-0.08,0.11,-0.08,0.10,ns,ns


In [104]:
# H3 – Country-specific correlations with p-values (Pearson + Spearman)
h3_country_p_rows = []
for country, group in matched.groupby('cost_country'):
    if group['cost_total_usd'].nunique() < 2:
        continue
    for metric in ['gpa_4_standardized', 'IELTS']:
        subset = group[['cost_total_usd', metric]].dropna()
        if len(subset) < 100 or subset[metric].nunique() < 2:
            continue
        r_p, p_p = sp_stats.pearsonr(subset[metric], subset['cost_total_usd'])
        r_s, p_s = sp_stats.spearmanr(subset[metric], subset['cost_total_usd'])
        h3_country_p_rows.append({
            'country': country, 'metric': metric, 'n': len(subset),
            'pearson_r': round(r_p, 4), 'pearson_p': p_p,
            'spearman_rho': round(r_s, 4), 'spearman_p': p_s,
            'pearson_sig': '***' if p_p < 0.001 else ('**' if p_p < 0.01 else ('*' if p_p < 0.05 else 'ns')),
            'spearman_sig': '***' if p_s < 0.001 else ('**' if p_s < 0.01 else ('*' if p_s < 0.05 else 'ns')),
        })

h3_country_p_df = pd.DataFrame(h3_country_p_rows)
h3_country_p_df

,country,metric,n,pearson_r,pearson_p,spearman_rho,spearman_p,pearson_sig,spearman_sig
0,Australia,gpa_4_standardized,2525,0.14,0.00,0.14,0.00,***,***
1,Australia,IELTS,960,0.07,0.03,0.04,0.28,*,ns
2,Singapore,gpa_4_standardized,4829,-0.12,0.00,-0.15,0.00,***,***
3,Singapore,IELTS,4239,-0.06,0.00,-0.05,0.00,***,***
4,UK,gpa_4_standardized,10348,0.07,0.00,0.15,0.00,***,***
5,UK,IELTS,5280,0.08,0.00,0.11,0.00,***,***
6,USA,gpa_4_standardized,465,0.10,0.03,0.04,0.42,*,ns
7,USA,IELTS,204,-0.05,0.52,-0.08,0.25,ns,ns


In [105]:
# H3 – Within-country OLS: cost ~ GPA_z + IELTS_z + tier dummies (numpy, no statsmodels)
# Standardised coefficients let us compare GPA vs IELTS effect sizes within each country.
def ols_tstat(y, X, feat_names):
    n, k = X.shape
    coeffs, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ coeffs
    ss_res = float(np.dot(resid, resid))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = 1 - ss_res / ss_tot if ss_tot > 1e-10 else np.nan
    mse = ss_res / max(n - k, 1)
    cov = mse * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(cov).clip(0))
    t_stat = np.where(se > 1e-10, coeffs / se, np.nan)
    p_vals = 2 * sp_stats.t.sf(np.abs(t_stat), df=max(n - k, 1))
    return pd.DataFrame({'feature': feat_names, 'coeff': coeffs,
                         'se': se, 't': t_stat, 'p': p_vals}), r2, n

ols_rows = []
for country, group in matched.groupby('cost_country'):
    if group['cost_total_usd'].nunique() < 3:
        continue
    g = group[['cost_total_usd', 'gpa_4_standardized', 'IELTS', 'undergrad_tier']].dropna()
    if len(g) < 50:
        continue
    y = g['cost_total_usd'].values
    gpa_z = (g['gpa_4_standardized'] - g['gpa_4_standardized'].mean()) / (g['gpa_4_standardized'].std() or 1)
    ielts_z = (g['IELTS'] - g['IELTS'].mean()) / (g['IELTS'].std() or 1)
    is_211 = (g['undergrad_tier'] == '211').astype(float).values
    is_other = (g['undergrad_tier'] == 'Other').astype(float).values
    X = np.column_stack([np.ones(len(g)), gpa_z.values, ielts_z.values, is_211, is_other])
    coeff_df, r2, n_obs = ols_tstat(y, X, ['intercept', 'gpa_z', 'ielts_z', 'tier_211', 'tier_Other'])
    for _, row in coeff_df.iterrows():
        ols_rows.append({
            'country': country, 'n': n_obs, 'R2': round(r2, 4),
            'feature': row['feature'], 'coeff': round(row['coeff'], 2),
            'se': round(row['se'], 2), 't': round(row['t'], 3), 'p': row['p'],
            'sig': '***' if row['p'] < 0.001 else ('**' if row['p'] < 0.01 else ('*' if row['p'] < 0.05 else 'ns'))
        })

h3_ols_df = pd.DataFrame(ols_rows)
h3_ols_df[h3_ols_df['feature'] != 'intercept'].round(4)

,country,n,R2,feature,coeff,se,t,p,sig
1,Australia,955,0.03,gpa_z,"1,042.95",325.20,3.21,0.00,**
2,Australia,955,0.03,ielts_z,562.31,326.06,1.73,0.08,ns
3,Australia,955,0.03,tier_211,"2,811.47",924.24,3.04,0.00,**
4,Australia,955,0.03,tier_Other,207.35,860.52,0.24,0.81,ns
6,Singapore,4217,0.02,gpa_z,"-7,096.77",795.35,-8.92,0.00,***
7,Singapore,4217,0.02,ielts_z,"-1,482.55",792.75,-1.87,0.06,ns
8,Singapore,4217,0.02,tier_211,"1,930.99","1,790.28",1.08,0.28,ns
9,Singapore,4217,0.02,tier_Other,"4,811.37","2,087.79",2.31,0.02,*
11,UK,5259,0.02,gpa_z,"3,140.18",469.67,6.69,0.00,***
12,UK,5259,0.02,ielts_z,"1,750.21",474.70,3.69,0.00,***


**Interpretation for RQ3.**

The full correlation table (Pearson + Spearman, with p-values) shows that GPA and IELTS are both **statistically significant** predictors of destination cost overall (both p < 0.001), but the effect sizes are very small (r < 0.10). Statistical significance here is driven by sample size (~12 000–20 000 rows), not practical importance.

The country-specific correlation table clarifies the picture. For **UK and Australia**, both Pearson and Spearman correlations are positive and significant — higher GPA and IELTS are weakly associated with higher-cost destinations. For **Singapore**, the correlations are negative and significant — higher academic scores are associated with *lower*-cost schools, likely because NUS (cheaper than NTU) attracts higher-scoring applicants.

The within-country OLS regression (cost ~ GPA_z + IELTS_z + tier dummies) confirms that after accounting for tier background, GPA and IELTS each contribute modest but often statistically significant positive effects in the UK and Australia, while the direction reverses in Singapore. **H3 is partially supported**: academic indicators are associated with destination cost, but the sign and strength vary substantially by country, exactly as H3 predicts.

## 5. Statistical Test Summary

The table below consolidates the key inferential results across all three hypotheses.

In [106]:
test_summary = pd.DataFrame([
    {'H': 'H1', 'Test': 'Spearman rho (school-level offer count vs cost)',
     'Result': f'rho={rho_s_h1:.3f}, p={p_s_h1:.3g}',
     'Supported': 'No (negative, not positive)'},
    {'H': 'H1', 'Test': 'Bootstrap 95% CI for Pearson r',
     'Result': f'[{ci_low_h1:.3f}, {ci_high_h1:.3f}]',
     'Supported': 'No (CI excludes zero on negative side)'},
    {'H': 'H1', 'Test': 'Kruskal-Wallis (cost across frequency bands)',
     'Result': f'H={kw_h1:.2f}, p={kw_p_h1:.3g}',
     'Supported': 'No (high-freq schools are cheaper)'},
    {'H': 'H2', 'Test': 'Chi-square (tier x cost quartile)',
     'Result': f'chi2={chi2_val:.1f}, p<0.001, V={cramers_v:.3f}',
     'Supported': 'Yes'},
    {'H': 'H2', 'Test': 'Kruskal-Wallis (cost across tiers)',
     'Result': f'H={kw_h2:.1f}, p<0.001',
     'Supported': 'Yes'},
    {'H': 'H2', 'Test': 'Within-country Kruskal-Wallis (tier effect net of country)',
     'Result': 'Significant in UK and Australia',
     'Supported': 'Yes (partial)'},
    {'H': 'H3', 'Test': 'Pearson / Spearman GPA vs cost (overall)',
     'Result': 'r~0.07, rho~0.09, both p<0.001',
     'Supported': 'Weakly (small effect)'},
    {'H': 'H3', 'Test': 'Country-specific correlations with p-values',
     'Result': 'UK/AU positive, SG negative (all sig.)',
     'Supported': 'Partially (direction varies by country)'},
    {'H': 'H3', 'Test': 'Within-country OLS (cost ~ GPA_z + IELTS_z + tier)',
     'Result': 'UK/AU: GPA_z positive sig; SG: negative sig',
     'Supported': 'Partially'},
])
test_summary

,H,Test,Result,Supported
0,H1,Spearman rho (school-level offer count vs cost),"rho=-0.508, p=2.89e-05","No (negative, not positive)"
1,H1,Bootstrap 95% CI for Pearson r,"[-0.502, 0.044]",No (CI excludes zero on negative side)
2,H1,Kruskal-Wallis (cost across frequency bands),"H=16.39, p=0.000943",No (high-freq schools are cheaper)
3,H2,Chi-square (tier x cost quartile),"chi2=1779.4, p<0.001, V=0.212",Yes
4,H2,Kruskal-Wallis (cost across tiers),"H=981.0, p<0.001",Yes
5,H2,Within-country Kruskal-Wallis (tier effect net...,Significant in UK and Australia,Yes (partial)
6,H3,Pearson / Spearman GPA vs cost (overall),"r~0.07, rho~0.09, both p<0.001",Weakly (small effect)
7,H3,Country-specific correlations with p-values,"UK/AU positive, SG negative (all sig.)",Partially (direction varies by country)
8,H3,Within-country OLS (cost ~ GPA_z + IELTS_z + t...,UK/AU: GPA_z positive sig; SG: negative sig,Partially


## 6. Overall Conclusion

Taken together, the linked dataset suggests three main conclusions.

1. **H1 is not supported in a simple school-level analysis.** More frequently appearing offer destinations are not, on average, the most expensive destinations.
2. **H2 receives the strongest support.** Applicants from `985` institutions are the most concentrated in higher-cost destinations, `211` applicants fall in the middle, and `Other` applicants are least concentrated in the highest cost quartile.
3. **H3 is mixed and country-specific.** GPA and language scores show some relationship with destination cost, but the relationship is modest and varies across destination countries.

This is exactly the value of linking the two datasets: the offer data alone cannot answer questions about affordability, and the cost data alone cannot show which kinds of applicants are reaching more expensive destinations.

## 7. Limitations

- The analysis only includes rows that could be matched to the cost dataset.
- `cost_total_usd` is a representative school-level estimate rather than a customized cost for each exact applicant-program combination.
- Undergraduate institution tier is based on the provided `985` and `211` school lists, which is transparent and reproducible but still only one way to operationalize applicant background.
- TOEFL, GRE, and GMAT are much sparser than GPA and IELTS, so they are better treated as supplementary evidence.
- Some destination systems, especially Hong Kong and Singapore, have relatively few distinct cost values in the matched sample, which limits how much can be learned from within-country correlation alone.

These limitations do not invalidate the analysis, but they should be stated clearly when interpreting the results.